In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from io import BytesIO
from PIL import Image

import tensorflow as tf
from tensorflow import keras


upl = files.upload()
img = Image.open(BytesIO(upl['img.jpg']))
img_style = Image.open(BytesIO(upl['img_style.jpg']))

plt.subplot(1, 2, 1)
plt.imshow( img )
plt.subplot(1, 2, 2)
plt.imshow( img_style )
plt.show()

x_img = keras.applications.vgg19.preprocess_input( np.expand_dims(img, axis=0) )
x_style = keras.applications.vgg19.preprocess_input(np.expand_dims(img_style, axis=0))

def deprocess_img(processed_img):
  x = processed_img.copy()
  if len(x.shape) == 4:
    x = np.squeeze(x, 0)
  assert len(x.shape) == 3, ("Input to deprocess image must be an image of "
                             "dimension [1, height, width, channel] or [height, width, channel]")
  if len(x.shape) != 3:
    raise ValueError("Invalid input to deprocessing image")
  
  # perform the inverse of the preprocessing step
  x[:, :, 0] += 103.939
  x[:, :, 1] += 116.779
  x[:, :, 2] += 123.68
  x = x[:, :, ::-1]

  x = np.clip(x, 0, 255).astype('uint8')
  return x

vgg = keras.applications.vgg19.VGG19(include_top=False, weights='imagenet')
vgg.trainable = False 

# Content layer where will pull our feature maps
content_layers = ['block5_conv2'] 

# Style layer we are interested in
style_layers = ['block1_conv1',
                'block2_conv1',
                'block3_conv1', 
                'block4_conv1', 
                'block5_conv1'
               ]

num_content_layers = len(content_layers)
num_style_layers = len(style_layers)

style_outputs = [vgg.get_layer(name).output for name in style_layers]
content_outputs = [vgg.get_layer(name).output for name in content_layers]
model_outputs = style_outputs + content_outputs

print(vgg.input)
for m in model_outputs:
  print(m)

model = keras.models.Model(vgg.input, model_outputs)
for layer in model.layers:
    layer.trainable = False

print(model.summary())      # вывод структуры НС в консоль

def get_feature_representations(model):
  # batch compute content and style features
  style_outputs = model(x_style)
  content_outputs = model(x_img)
  
  # Get the style and content feature representations from our model  
  style_features = [style_layer[0] for style_layer in style_outputs[:num_style_layers]]
  content_features = [content_layer[0] for content_layer in content_outputs[num_style_layers:]]
  return style_features, content_features

def get_content_loss(base_content, target):
  return tf.reduce_mean(tf.square(base_content - target))

def gram_matrix(input_tensor):
  # We make the image channels first 
  channels = int(input_tensor.shape[-1])
  a = tf.reshape(input_tensor, [-1, channels])
  n = tf.shape(a)[0]
  gram = tf.matmul(a, a, transpose_a=True)
  return gram / tf.cast(n, tf.float32)

def get_style_loss(base_style, gram_target):
  gram_style = gram_matrix(base_style)
  
  return tf.reduce_mean(tf.square(gram_style - gram_target))

def compute_loss(model, loss_weights, init_image, gram_style_features, content_features):
  style_weight, content_weight = loss_weights
  
  model_outputs = model(init_image)
  
  style_output_features = model_outputs[:num_style_layers]
  content_output_features = model_outputs[num_style_layers:]
  
  style_score = 0
  content_score = 0

  # Accumulate style losses from all layers
  # Here, we equally weight each contribution of each loss layer
  weight_per_style_layer = 1.0 / float(num_style_layers)
  for target_style, comb_style in zip(gram_style_features, style_output_features):
    style_score += weight_per_style_layer * get_style_loss(comb_style[0], target_style)
    
  # Accumulate content losses from all layers 
  weight_per_content_layer = 1.0 / float(num_content_layers)
  for target_content, comb_content in zip(content_features, content_output_features):
    content_score += weight_per_content_layer* get_content_loss(comb_content[0], target_content)
  
  style_score *= style_weight
  content_score *= content_weight

  # Get total loss
  loss = style_score + content_score 
  return loss, style_score, content_score

num_iterations=100
content_weight=1e3
style_weight=1e-2

style_features, content_features = get_feature_representations(model)
gram_style_features = [gram_matrix(style_feature) for style_feature in style_features]

init_image = np.copy(x_img)
init_image = tf.Variable(init_image, dtype=tf.float32)

opt = tf.compat.v1.train.AdamOptimizer(learning_rate=2, beta1=0.99, epsilon=1e-1)
iter_count = 1
best_loss, best_img = float('inf'), None
loss_weights = (style_weight, content_weight)

cfg = {
      'model': model,
      'loss_weights': loss_weights,
      'init_image': init_image,
      'gram_style_features': gram_style_features,
      'content_features': content_features
}

norm_means = np.array([103.939, 116.779, 123.68])
min_vals = -norm_means
max_vals = 255 - norm_means
imgs = []

for i in range(num_iterations):
    with tf.GradientTape() as tape: 
       all_loss = compute_loss(**cfg)
    
    loss, style_score, content_score = all_loss
    grads = tape.gradient(loss, init_image)

    opt.apply_gradients([(grads, init_image)])
    clipped = tf.clip_by_value(init_image, min_vals, max_vals)
    init_image.assign(clipped)
    
    if loss < best_loss:
      # Update best loss and best image from total loss. 
      best_loss = loss
      best_img = deprocess_img(init_image.numpy())

      # Use the .numpy() method to get the concrete numpy array
      plot_img = deprocess_img(init_image.numpy())
      imgs.append(plot_img)
      print('Iteration: {}'.format(i))

plt.imshow(best_img)
print(best_loss)

image = Image.fromarray(best_img.astype('uint8'), 'RGB')
image.save("result.jpg")
files.download("result.jpg")

## Izah

In [ ]:
import numpy as np
# NumPy — ədədlər və çoxölçülü array-lərlə işləmək üçün əsas kitabxanadır.
#
# Bu proqramda NumPy-dan:
# - şəklə "batch" ölçüsü əlavə etmək üçün (np.expand_dims)
# - batch ölçüsünü geri silmək üçün (np.squeeze)
# - qiymətləri 0–255 aralığında saxlamaq üçün (np.clip)
# - orta piksel qiymətlərini array kimi saxlamaq üçün (np.array)
# istifadə olunur.


import matplotlib.pyplot as plt
# matplotlib.pyplot — şəkilləri və qrafikləri ekranda göstərmək üçündür.
#
# Burada plt:
# - başlanğıc şəkilləri yan-yana göstərmək üçün (subplot + imshow)
# - son nəticəni göstərmək üçün istifadə olunur.


from google.colab import files
# Colab-da fayl yükləmək (upload) və yükləmək (download) üçün modul.
#
# DİQQƏT: Bu kod yalnız Google Colab-da işləyir.


from io import BytesIO
# BytesIO — baytları yaddaşda "virtual fayl" kimi oxunan obyektə çevirir.
# Image.open() fayl kimi obyekt gözlədiyi üçün lazımdır.


from PIL import Image
# PIL (Pillow) — şəkilləri açmaq, çevirmək və saxlamaq üçün kitabxana.


import tensorflow as tf
# TensorFlow-un özünü tf adı ilə import edirik.
#
# ƏVVƏLKİ KODLARDAN FƏRQ:
# Əvvəl yalnız "from tensorflow import keras" yazırdıq.
# Bu dəfə birbaşa tf lazımdır, çünki aşağıda aşağı səviyyəli
# əməliyyatlardan istifadə edəcəyik:
# - tf.GradientTape  → qradiyent hesablamaq üçün
# - tf.Variable      → dəyişdirilə bilən tensor yaratmaq üçün
# - tf.reshape, tf.matmul, tf.square, tf.reduce_mean və s.


from tensorflow import keras
# Keras API — burada hazır VGG19 modelini yükləmək üçün istifadə olunur.


# =====================================================================
# BU KOD NƏDİR? — NEURAL STYLE TRANSFER (üslub köçürmə)
# =====================================================================
#
# Məqsəd: iki şəkildən istifadə edib üçüncü şəkil yaratmaq:
#
# 1. img        → MƏZMUN şəkli (nəyin təsvir olunduğu: insan, bina, mənzərə)
# 2. img_style  → ÜSLUB şəkli (rəng, fırça vuruşları, tekstura:
#                 məsələn Van Qoqun rəsmi)
# 3. result.jpg → nəticə: məzmun birinci şəkildən, üslub ikinci şəkildən
#
# ƏN VACİB FƏRQ (əvvəlki kodlarla müqayisədə):
#
# MNIST kodunda:  model öyrədilirdi (fit), şəkil dəyişmirdi.
# VGG16 kodunda:  hazır model proqnoz verirdi (predict).
# Bu kodda:       MODELİN ÇƏKİLƏRİ DƏYİŞMİR (dondurulub),
#                 əvəzinə ŞƏKLİN ÖZÜ piksel-piksel dəyişdirilir.
#
# Yəni "model nəyi öyrənir?" yox, "şəkil necə dəyişsin ki,
# loss kiçilsin?" sualı verilir.


upl = files.upload()
# Colab-da fayl seçmə pəncərəsi açır.
#
# Burada İKİ şəkil yükləməlisən:
# - img.jpg        (məzmun şəkli)
# - img_style.jpg  (üslub şəkli)
#
# Nəticə upl dəyişənində lüğət (dictionary) kimi saxlanılır:
# açar → faylın adı, qiymət → faylın baytları.
#
# Fayl adları dəqiq 'img.jpg' və 'img_style.jpg' olmalıdır.
# Əks halda aşağıda KeyError xətası çıxacaq.


img = Image.open(BytesIO(upl['img.jpg']))
img_style = Image.open(BytesIO(upl['img_style.jpg']))
# Hər iki şəkli PIL şəkli kimi açırıq.
#
# upl['img.jpg']  → lüğətdən faylın baytlarını götürür
# BytesIO(...)    → baytları fayl kimi oxunan obyektə çevirir
# Image.open(...) → şəkli açır
#
# DİQQƏT: Burada resize yoxdur, şəkillər öz orijinal ölçüsündədir.
# VGG19 include_top=False olduğu üçün istənilən ölçünü qəbul edir,
# amma şəkil böyükdürsə proses çox yavaş olar və yaddaş (RAM/GPU)
# çatışmaya bilər. Buna görə şəkilləri təxminən 512 piksel və
# ya daha kiçik götürmək məsləhətdir.
#
# Şəkillər 3 kanallı (RGB) olmalıdır. PNG-də şəffaflıq (4-cü kanal)
# varsa, xəta çıxar. Belə halda: img = img.convert('RGB')


plt.subplot(1, 2, 1)
plt.imshow( img )
plt.subplot(1, 2, 2)
plt.imshow( img_style )
plt.show()
# İki şəkli yan-yana göstərir ki, düzgün yükləndiyini yoxlayaq.
#
# plt.subplot(1, 2, 1)
# → 1 sətir, 2 sütunluq şəbəkə, birinci xana (sol).
#
# plt.subplot(1, 2, 2)
# → eyni şəbəkənin ikinci xanası (sağ).
#
# Solda məzmun şəkli, sağda üslub şəkli görünəcək.
#
# plt.show() → şəkilləri ekrana çıxarır.


x_img = keras.applications.vgg19.preprocess_input( np.expand_dims(img, axis=0) )
x_style = keras.applications.vgg19.preprocess_input(np.expand_dims(img_style, axis=0))
# Hər iki şəkli VGG19-un tələb etdiyi formata salır.
#
# İçəridən çölə doğru iki addım var:
#
# 1. np.expand_dims(img, axis=0)
#    → PIL şəklini avtomatik array-ə çevirir və başa batch ölçüsü əlavə edir:
#      (hündürlük, en, 3)  →  (1, hündürlük, en, 3)
#
# 2. keras.applications.vgg19.preprocess_input(...)
#    → VGG16 kodundakı ilə eyni işi görür:
#      a) RGB → BGR
#      b) hər kanaldan ImageNet-in orta qiymətini çıxır
#         (B: 103.939, G: 116.779, R: 123.68)
#
# Nəticədə x_img və x_style:
# - forması (1, H, W, 3)
# - qiymətləri 0 ətrafında (mənfi də ola bilər)
#
# Qeyd: VGG16 kodunda vgg16.preprocess_input idi,
# burada vgg19.preprocess_input-dur, çünki VGG19 modelindən istifadə edirik.
# (Hər ikisi eyni işi görür.)


def deprocess_img(processed_img):
  # Bu funksiya preprocess_input-un TƏRSİNİ edir.
  #
  # Niyə lazımdır?
  # Optimizasiya zamanı şəkil VGG formatında (BGR, orta çıxılmış)
  # saxlanılır. Bu formatda şəkli ekranda göstərə və ya fayla yaza bilmərik.
  # Ona görə onu yenidən normal RGB şəklə (0–255) qaytarırıq.

  x = processed_img.copy()
  # Orijinal array-i pozmamaq üçün surətini çıxarırıq.

  if len(x.shape) == 4:
    x = np.squeeze(x, 0)
  # Forma 4 ölçülüdürsə (1, H, W, 3), batch ölçüsünü silirik:
  #
  # (1, H, W, 3)  →  (H, W, 3)
  #
  # np.squeeze(x, 0) → 0-cı oxdakı ölçünü (ölçüsü 1 olan) silir.
  # Bu expand_dims(axis=0)-ın tərsidir.

  assert len(x.shape) == 3, ("Input to deprocess image must be an image of "
                             "dimension [1, height, width, channel] or [height, width, channel]")
  # assert → şərt pozularsa, proqram xəta mesajı ilə dayanır.
  # Burada yoxlayır: indi forma dəqiq 3 ölçülüdürmü?

  if len(x.shape) != 3:
    raise ValueError("Invalid input to deprocessing image")
  # Eyni yoxlamanın ikinci variantı. Praktikada yuxarıdakı assert artıq
  # bunu tutur, ona görə bu sətr heç vaxt işləməyəcək (təkrardır).

  # perform the inverse of the preprocessing step
  # Yuxarıdakı ingilis dilindəki şərh: "preprocessing addımının tərsini edirik".
  x[:, :, 0] += 103.939
  x[:, :, 1] += 116.779
  x[:, :, 2] += 123.68
  # Hər kanala çıxılmış orta qiyməti geri əlavə edirik.
  #
  # Kanal 0 → Mavi (B)   + 103.939
  # Kanal 1 → Yaşıl (G)  + 116.779
  # Kanal 2 → Qırmızı (R) + 123.68
  #
  # (Hələ BGR sırasındayıq, ona görə B birinci gəlir.)

  x = x[:, :, ::-1]
  # Kanalların sırasını tərsinə çevirir: BGR → RGB.
  #
  # ::-1 → "axıradan əvvələ doğru" deməkdir.
  # Beləliklə (B, G, R) → (R, G, B).

  x = np.clip(x, 0, 255).astype('uint8')
  # np.clip(x, 0, 255)
  # → 0-dan kiçik qiymətləri 0, 255-dən böyükləri 255 edir.
  #
  # .astype('uint8')
  # → qiymətləri 0–255 aralığında tam ədədə çevirir.
  #   Şəkil üçün standart format budur.

  return x
  # Hazır RGB şəkil array-ini qaytarır. Forması (H, W, 3).


vgg = keras.applications.vgg19.VGG19(include_top=False, weights='imagenet')
# Hazır VGG19 modelini yükləyirik.
#
# VGG19 — VGG16-nın daha dərin versiyasıdır (19 çəkili qat).
#
# weights='imagenet'
# → ImageNet-də ARTIQ öyrədilmiş çəkilərdən istifadə edir.
#
# include_top=False   ← VGG16 kodundan ƏSAS FƏRQ
# → modelin sonundakı Dense qatlarını (təsnifat hissəsini) ATIR.
# → Yalnız Conv və Pooling qatları qalır.
#
# Niyə?
# Bizə şəkli siniflərə ayırmaq lazım deyil.
# Bizə VGG-nin ara qatlarında şəkildən çıxardığı XÜSUSİYYƏTLƏR lazımdır.
#
# Bonus: Dense qat olmadığı üçün giriş ölçüsü sabit (224×224) deyil,
# istənilən ölçülü şəkil verilə bilər.


vgg.trainable = False
# VGG-nin çəkilərini dondurur.
#
# Bu, modelin heç bir çəkisinin dəyişməyəcəyini bildirir.
# Model sadəcə "xüsusiyyət çıxarıcı" kimi işləyir.
#
# Dəyişən yeganə şey — şəklin özü olacaq.


# Content layer where will pull our feature maps
# Yuxarıdakı ingilis dilindəki şərh: "xüsusiyyət xəritələrini götürəcəyimiz məzmun qatı".
content_layers = ['block5_conv2']
# MƏZMUN üçün istifadə olunan qat.
#
# block5_conv2 → VGG19-un 5-ci blokunun 2-ci konvolyusiya qatı.
#
# Bu qat şəbəkənin DƏRİN hissəsindədir.
# Dərin qatlar şəkildəki yüksək səviyyəli məzmunu "bilir"
# (burada bina var, burada insan var), amma dəqiq piksel
# detallarını yox.
#
# Məzmun loss-u bu qatın çıxışına əsasən hesablanacaq.


# Style layer we are interested in
# Yuxarıdakı ingilis dilindəki şərh: "maraqlandığımız üslub qatları".
style_layers = ['block1_conv1',
                'block2_conv1',
                'block3_conv1',
                'block4_conv1',
                'block5_conv1'
               ]
# ÜSLUB üçün istifadə olunan qatlar — hər blokun BİRİNCİ konvolyusiya qatı.
#
# Niyə birdən çox qat?
# Üslub müxtəlif miqyaslarda mövcuddur:
#
# - dayaz qatlar (block1, block2) → xırda detallar: rəng, kiçik xətlər, tekstura
# - dərin qatlar (block4, block5)  → iri formalar, daha mücərrəd naxışlar
#
# Hamısını birlikdə istifadə edəndə üslub tam tutulur.


num_content_layers = len(content_layers)
num_style_layers = len(style_layers)
# Qatların sayını saxlayırıq.
#
# num_content_layers → 1
# num_style_layers   → 5
#
# Bu rəqəmlər aşağıda iki işdə lazım olacaq:
# - model çıxışlarını üslub və məzmun hissələrinə bölmək üçün
# - loss-ları qat sayına bölüb orta almaq üçün


style_outputs = [vgg.get_layer(name).output for name in style_layers]
content_outputs = [vgg.get_layer(name).output for name in content_layers]
model_outputs = style_outputs + content_outputs
# Seçdiyimiz qatların çıxışlarını (output tensorlarını) toplayırıq.
#
# vgg.get_layer(name) → adına görə qatı tapır
# .output             → həmin qatın çıxışı
#
# style_outputs   → 5 tensor (5 üslub qatından)
# content_outputs → 1 tensor (1 məzmun qatından)
#
# model_outputs = style_outputs + content_outputs
# → sadə list birləşdirməsidir, cəmi 6 tensor.
#
# SIRA VACİBDİR:
#
# model_outputs[0:5] → üslub qatları
# model_outputs[5:]  → məzmun qatı
#
# Aşağıda [:num_style_layers] və [num_style_layers:] ilə
# məhz bu sıraya əsaslanıb bölürük.


print(vgg.input)
for m in model_outputs:
  print(m)
# Yoxlama məqsədli çaplar.
#
# print(vgg.input) → modelin giriş tensorunu göstərir.
# for döngüsü      → 6 çıxış tensorunun hər birini göstərir
#                    (adı və forması ilə).
#
# Bu, düzgün qatları seçdiyimizi yoxlamaq üçündür.
# Proqramın işinə təsir etmir.


model = keras.models.Model(vgg.input, model_outputs)
# Yeni bir model yaradırıq.
#
# Keras.models.Model(giriş, çıxışlar)
#
# Giriş:   VGG-nin girişi (şəkil)
# Çıxışlar: 6 ara qatın çıxışı
#
# Bu yeni model şəkli qəbul edir və 6 ədəd xüsusiyyət xəritəsi qaytarır
# (bir siyahı şəklində). Son nəticə kimi ehtimal deyil, ara xüsusiyyətlər.
#
# ƏVVƏLKİ KODLARDAN FƏRQ:
# Burada Sequential yox, "funksional API" istifadə olunur.
# Çünki çoxsaylı çıxış lazımdır, Sequential isə yalnız bir çıxış verir.


for layer in model.layers:
    layer.trainable = False
# Hər ehtiyat üçün yeni modelin bütün qatlarını bir daha dondururuq.
# (Yuxarıda vgg.trainable = False artıq bunu edib,
# bu təkrar təminatdır.)


print(model.summary())      # вывод структуры НС в консоль
# Yuxarıdakı rus dilindəki şərh: "neyron şəbəkəsinin strukturunu konsola çıxarmaq".
#
# Modelin strukturunu çap edir: hər qatın adı, çıxış forması, parametr sayı.
#
# Qeyd: model.summary() özü çap edir və None qaytarır.
# Ona görə print(...) içinə yazmaq artıqdır,
# sonda əlavə "None" sözü də çap olunacaq.
# Sadəcə model.summary() yazmaq kifayətdir.


def get_feature_representations(model):
  # Bu funksiya hər iki şəkli modeldən keçirir və lazım olan
  # xüsusiyyətləri çıxarır.
  #
  # Bunu yalnız BİR DƏFƏ, optimizasiyadan əvvəl edirik, çünki
  # üslub şəklinin və məzmun şəklinin xüsusiyyətləri dəyişmir.
  # Bunlar "hədəf"dir, şəklin yaxınlaşmağa çalışdığı qiymətlərdir.

  # batch compute content and style features
  # Yuxarıdakı ingilis dilindəki şərh: "məzmun və üslub xüsusiyyətlərini hesabla".
  style_outputs = model(x_style)
  content_outputs = model(x_img)
  # Model hər dəfə 6 tensordan ibarət siyahı qaytarır.
  #
  # style_outputs   → üslub şəklinin 6 qatdakı çıxışları
  # content_outputs → məzmun şəklinin 6 qatdakı çıxışları
  #
  # Hər ikisindən yalnız lazım olan hissə aşağıda götürülür.

  # Get the style and content feature representations from our model
  # Yuxarıdakı ingilis dilindəki şərh: "modeldən üslub və məzmun xüsusiyyətlərini alırıq".
  style_features = [style_layer[0] for style_layer in style_outputs[:num_style_layers]]
  content_features = [content_layer[0] for content_layer in content_outputs[num_style_layers:]]
  # style_outputs[:num_style_layers]
  # → ilk 5 çıxış (üslub qatları)
  #
  # content_outputs[num_style_layers:]
  # → 5-ci indeksdən sonrakı hissə (məzmun qatı)
  #
  # [0]
  # → hər tensordan batch ölçüsünü silir:
  #   (1, h, w, kanal)  →  (h, w, kanal)
  #
  # Qeyd: style_features üslub şəklindən, content_features
  # məzmun şəklindən götürülür. Hər şəklin "digər" çıxışları atılır.

  return style_features, content_features
  # İki siyahı qaytarır:
  # - style_features   → 5 ədəd (üslub şəklinin xüsusiyyətləri)
  # - content_features → 1 ədəd (məzmun şəklinin xüsusiyyəti)


def get_content_loss(base_content, target):
  # MƏZMUN loss-u.
  #
  # base_content → hazırkı yaradılan şəklin məzmun qatındakı xüsusiyyətləri
  # target       → orijinal məzmun şəklinin həmin qatdakı xüsusiyyətləri
  #
  # Sadə fikir: iki şəklin xüsusiyyətləri nə qədər yaxındırsa,
  # məzmun o qədər qorunub.
  return tf.reduce_mean(tf.square(base_content - target))
  # base_content - target → hər elementin fərqi
  # tf.square(...)        → fərqin kvadratı (mənfi işarə yoxa çıxır)
  # tf.reduce_mean(...)   → bütün kvadratların ortası
  #
  # Bu, Mean Squared Error (MSE), yəni orta kvadratik xətadır.
  # Fərq kiçik olduqca loss da kiçik olur. Fərq yoxdursa, loss = 0.


def gram_matrix(input_tensor):
  # GRAM MATRİSİ — üslubu riyazi olaraq ifadə etməyin yolu.
  #
  # Əsas fikir:
  # Üslub "NƏ harada var" deyil, "HANSI xüsusiyyətlər BİRLİKDƏ
  # görünür" deməkdir.
  #
  # Məsələn: "mavi rəng ilə qıvrım fırça vuruşları həmişə yan-yana gəlir".
  # Yerin əhəmiyyəti yoxdur, birgə görünmə statistikası vacibdir.
  #
  # Gram matrisi tam bunu ölçür: hər iki kanalın bir-biri ilə
  # nə qədər "korrelyasiya olduğunu" göstərir.

  # We make the image channels first
  # Yuxarıdakı ingilis dilindəki şərh: "şəkil kanallarını birinci edirik".
  # (Qeyd: kodda əslində kanalları birinci etmirik, yalnız reshape edirik.
  # Şərh bir qədər yanıltıcıdır.)
  channels = int(input_tensor.shape[-1])
  # Kanal (filter) sayı. Forma (h, w, kanal) olduğu üçün
  # axırıncı ölçü kanal sayıdır. Məsələn block1_conv1 üçün 64.

  a = tf.reshape(input_tensor, [-1, channels])
  # Forma dəyişir:
  #
  # (h, w, kanal)  →  (h*w, kanal)
  #
  # -1 → "bu ölçünü özün hesabla" deməkdir (h*w).
  # Yəni 2 ölçülü xüsusiyyət xəritəsi düzəldilib bir uzun sütuna çevrilir.
  # Hər sətir bir piksel mövqeyidir, hər sütun bir filtrdir.

  n = tf.shape(a)[0]
  # Sətirlərin sayı, yəni h*w (mövqe sayı).
  # Aşağıda normallaşdırma üçün istifadə olunacaq.

  gram = tf.matmul(a, a, transpose_a=True)
  # a-nın transpozunu a ilə vurur:
  #
  # aᵀ × a
  # (kanal, h*w) × (h*w, kanal)  =  (kanal, kanal)
  #
  # Nəticə kanal × kanal ölçülü matrisdir.
  # Matrisin [i, j] elementi i-ci və j-ci filterlərin
  # bütün mövqelərdə nə qədər birlikdə aktiv olduğunu göstərir.
  #
  # Mövqe məlumatı burada İTİRİLİR (h*w ölçüsü yox olur).
  # Məhz buna görə Gram matrisi "yerdən asılı olmayan" üslub təsvirini verir.

  return gram / tf.cast(n, tf.float32)
  # Mövqe sayına bölərək normallaşdırır.
  # Beləliklə böyük və kiçik şəkillərin Gram matrisləri müqayisə oluna bilir.
  #
  # tf.cast(n, tf.float32) → n-i ondalıq ədədə çevirir (bölmə üçün lazımdır).


def get_style_loss(base_style, gram_target):
  # ÜSLUB loss-u.
  #
  # base_style  → hazırkı yaradılan şəklin üslub qatındakı xüsusiyyətləri
  # gram_target → üslub şəklinin (əvvəlcədən hesablanmış) Gram matrisi
  gram_style = gram_matrix(base_style)
  # Yaradılan şəklin Gram matrisini hesablayırıq.
  # (Hədəf Gram matrisi artıq hazırdır.)

  return tf.reduce_mean(tf.square(gram_style - gram_target))
  # İki Gram matrisinin MSE-si.
  #
  # Fərq kiçik olduqca yaradılan şəklin üslubu
  # üslub şəklinin üslubuna yaxınlaşır.


def compute_loss(model, loss_weights, init_image, gram_style_features, content_features):
  # Ümumi loss-u hesablayır: üslub loss + məzmun loss.
  #
  # Parametrlər:
  # model               → 6 çıxışlı xüsusiyyət modeli
  # loss_weights        → (style_weight, content_weight)
  # init_image          → hazırda dəyişdirilən şəkil
  # gram_style_features → üslub şəklinin Gram matrisləri (hədəf)
  # content_features    → məzmun şəklinin xüsusiyyətləri (hədəf)

  style_weight, content_weight = loss_weights
  # Tuple-ı iki dəyişənə açırıq.

  model_outputs = model(init_image)
  # Hazırkı şəkli modeldən keçiririk.
  # Nəticədə 6 tensor: 5 üslub + 1 məzmun.

  style_output_features = model_outputs[:num_style_layers]
  content_output_features = model_outputs[num_style_layers:]
  # Çıxışları iki hissəyə bölürük (əvvəlki kimi, sıraya əsasən).

  style_score = 0
  content_score = 0
  # Loss-ları toplayacağımız dəyişənlər, sıfırdan başlayır.

  # Accumulate style losses from all layers
  # Here, we equally weight each contribution of each loss layer
  # Yuxarıdakı ingilis dilindəki şərh: "bütün qatlardan üslub loss-larını toplayırıq,
  # hər qatın töhfəsi bərabər çəkiyə malikdir".
  weight_per_style_layer = 1.0 / float(num_style_layers)
  # Hər üslub qatının çəkisi: 1 / 5 = 0.2
  # Beləliklə bütün qatlar bərabər əhəmiyyətlidir.

  for target_style, comb_style in zip(gram_style_features, style_output_features):
    style_score += weight_per_style_layer * get_style_loss(comb_style[0], target_style)
  # zip(...) iki siyahını qat-qat birləşdirir:
  #
  # target_style → həmin qat üçün üslub şəklinin Gram matrisi (hədəf)
  # comb_style   → həmin qat üçün yaradılan şəklin çıxışı
  #
  # comb_style[0] → batch ölçüsünü silir.
  #
  # Hər qatın loss-u 0.2-yə vurulub style_score-a əlavə edilir.

  # Accumulate content losses from all layers
  # Yuxarıdakı ingilis dilindəki şərh: "bütün qatlardan məzmun loss-larını toplayırıq".
  weight_per_content_layer = 1.0 / float(num_content_layers)
  # Bir məzmun qatı olduğu üçün çəki 1 / 1 = 1.0

  for target_content, comb_content in zip(content_features, content_output_features):
    content_score += weight_per_content_layer* get_content_loss(comb_content[0], target_content)
  # Eyni məntiq: yaradılan şəklin məzmun xüsusiyyətləri ilə
  # orijinal məzmun şəklinin xüsusiyyətlərini müqayisə edir.

  style_score *= style_weight
  content_score *= content_weight
  # Hər loss-u öz ÇƏKİSİ ilə vururuq.
  #
  # Bu çəkilər şəklin nəyə daha çox yaxın olacağını müəyyən edir.
  # Aşağıda style_weight=1e-2, content_weight=1e3 verilib.
  # Yəni məzmun çəkisi daha böyükdür. (Gram loss-u adətən
  # çox böyük ədədlər verdiyi üçün onu belə tarazlaşdırırlar.)

  # Get total loss
  # Yuxarıdakı ingilis dilindəki şərh: "ümumi loss-u alırıq".
  loss = style_score + content_score
  return loss, style_score, content_score
  # Üç qiymət qaytarır: ümumi loss, üslub hissəsi, məzmun hissəsi.
  # Optimizasiya yalnız ümumi loss ilə aparılır.
  # Digər ikisi izləmək üçündür.


num_iterations=100
# Optimizasiya addımlarının sayı.
# Hər iterasiyada şəkil bir az dəyişdirilir. Daha çox iterasiya
# adətən daha yaxşı nəticə verir, amma daha çox vaxt aparır.

content_weight=1e3
style_weight=1e-2
# Çəkilər.
#
# 1e3  = 1000
# 1e-2 = 0.01
#
# Nisbət: məzmun üslubdan 100 000 dəfə böyük çəkiyə malikdir,
# amma bu fərq Gram loss-unun böyük ədədləri ilə tarazlanır.
#
# Təcrübə üçün:
# - style_weight-i artır → nəticə daha çox üslub şəklinə oxşayacaq
# - content_weight-i artır → nəticə daha çox orijinal şəklə oxşayacaq


style_features, content_features = get_feature_representations(model)
# Hədəf xüsusiyyətləri bir dəfə hesablayırıq:
# - üslub şəklinin 5 qatdakı xüsusiyyətləri
# - məzmun şəklinin 1 qatdakı xüsusiyyəti

gram_style_features = [gram_matrix(style_feature) for style_feature in style_features]
# Üslub xüsusiyyətlərini Gram matrisinə çeviririk.
# Hədəf üslub artıq Gram matrisi formasındadır (5 ədəd matris).
#
# Bunu da bir dəfə edirik, çünki üslub şəkli dəyişmir.


init_image = np.copy(x_img)
init_image = tf.Variable(init_image, dtype=tf.float32)
# Dəyişdiriləcək şəkli yaradırıq.
#
# np.copy(x_img)
# → başlanğıc nöqtə kimi MƏZMUN ŞƏKLİNİN SURƏTİNİ götürürük.
#   (Alternativ: təsadüfi səs-küy. Amma məzmun şəklindən başlamaq
#   daha sürətli və daha yaxşı nəticə verir.)
#
# tf.Variable(...)
# → TensorFlow-da dəyişdirilə bilən (öyrənilən) dəyişən yaradır.
#   Qradiyentlər məhz bu dəyişənə görə hesablanacaq və
#   optimizer onu yeniləyəcək.
#
# MNIST kodunda optimizer modelin çəkilərini yeniləyirdi.
# Burada optimizer ŞƏKLİN PİKSELLƏRİNİ yeniləyir.


opt = tf.compat.v1.train.AdamOptimizer(learning_rate=2, beta1=0.99, epsilon=1e-1)
# Adam optimizer-i yaradırıq (burada köhnə tf.compat.v1 versiyası).
#
# MNIST kodunda 'adam' sətir kimi yazırdıq və compile() onu idarə edirdi.
# Burada isə fit() olmadığı üçün optimizeri özümüz yaradıb
# əl ilə çağırırıq.
#
# learning_rate=2
# → öyrənmə sürəti çox böyükdür (adətən 0.001 olur).
#   Bu normaldır, çünki piksel qiymətləri −120 ... +150 kimi
#   böyük aralıqdadır, kiçik addımlar çox yavaş olardı.
#
# beta1=0.99
# → Adam-ın "momentum" parametri. Əvvəlki addımların qradiyentlərini
#   nə qədər yadda saxladığını bildirir.
#
# epsilon=1e-1
# → sıfıra bölməni əngəlləyən kiçik ədəd. Burada standartdan çox böyükdür
#   (adətən 1e-7), bu da yeniləmələri sabitləşdirir.


iter_count = 1
# İterasiya sayğacı. Bu kodda aşağıda istifadə olunmur.

best_loss, best_img = float('inf'), None
# Ən yaxşı nəticəni izləmək üçün.
#
# best_loss = float('inf')
# → sonsuzluq. Hər real loss ondan kiçik olacaq,
#   ona görə birinci iterasiya avtomatik "ən yaxşı" olur.
#
# best_img = None
# → hələ ən yaxşı şəkil yoxdur.

loss_weights = (style_weight, content_weight)
# İki çəkini bir tuple-da birləşdiririk.
# Sıra vacibdir: compute_loss içində də eyni sıra ilə açılır.


cfg = {
      'model': model,
      'loss_weights': loss_weights,
      'init_image': init_image,
      'gram_style_features': gram_style_features,
      'content_features': content_features
}
# compute_loss funksiyasının bütün arqumentlərini lüğətdə toplayırıq.
#
# Aşağıda compute_loss(**cfg) yazanda, ** lüğəti
# açıb hər açarı həmin adlı arqumentə ötürür. Yəni belə yazmaqla eynidir:
#
# compute_loss(model=model, loss_weights=loss_weights, ...)
#
# Sadəcə daha qısa yazılışdır.


norm_means = np.array([103.939, 116.779, 123.68])
min_vals = -norm_means
max_vals = 255 - norm_means
imgs = []
# Piksellərin buraxıla bilən aralığını hesablayırıq.
#
# Preprocess-dən sonra hər kanalın qiyməti:
# - ən az:   0 − orta   = −orta
# - ən çox:  255 − orta = 255 − orta
#
# min_vals → [-103.939, -116.779, -123.68]
# max_vals → [ 151.061,  138.221,  131.32]
#
# Bu, kanal-kanal fərqli iki array-dir (hər kanalın öz aralığı var).
# Optimizasiya zamanı piksellər bu aralıqdan çıxarsa, geri qaytarılacaq.
#
# imgs = [] → aralıq nəticələri saxlamaq üçün boş siyahı
# (aşağıda doldurulur, amma sonra istifadə olunmur).


for i in range(num_iterations):
    # 100 dəfə təkrarlanan əsas optimizasiya döngüsü.
    # Hər iterasiya = şəklin bir addım yaxşılaşdırılması.

    with tf.GradientTape() as tape:
       all_loss = compute_loss(**cfg)
    # tf.GradientTape — "qradiyent lentidir".
    # with blokunun içindəki bütün hesablamaları qeyd edir ki,
    # sonra qradiyenti (törəməni) hesablamaq mümkün olsun.
    #
    # Burada loss hesablanır: şəkil modeldən keçir,
    # üslub və məzmun loss-ları tapılır.
    #
    # Keras fit()-də bunu avtomatik edirdi.
    # Burada əl ilə edirik, çünki şəkli optimizasiya edirik, modeli yox.

    loss, style_score, content_score = all_loss
    # compute_loss üç qiymət qaytarır, onları ayırırıq.

    grads = tape.gradient(loss, init_image)
    # Qradiyenti hesablayır:
    # "loss-u azaltmaq üçün hər pikseli hansı istiqamətdə dəyişmək lazımdır?"
    #
    # Nəticə init_image ilə eyni formada olur (1, H, W, 3).
    # Hər pikselin hər kanalı üçün bir qradiyent qiyməti var.
    #
    # Qradiyent modelin çəkilərinə görə yox, ŞƏKLƏ görə hesablanır
    # (çünki çəkilər dondurulub).

    opt.apply_gradients([(grads, init_image)])
    # Optimizer şəkli yeniləyir.
    #
    # [(grads, init_image)] → (qradiyent, dəyişən) cütlərinin siyahısı.
    # Adam qradiyentə əsasən init_image-in pikselini bir az dəyişir
    # ki, loss kiçilsin.

    clipped = tf.clip_by_value(init_image, min_vals, max_vals)
    init_image.assign(clipped)
    # Piksel qiymətlərini yuxarıda hesabladığımız aralığa salır.
    #
    # Yeniləmədən sonra bəzi piksellər icazə verilən aralıqdan
    # çıxa bilər (məsələn 255-dən böyük ola bilər).
    # Bunları sərhədə qaytarırıq.
    #
    # init_image.assign(clipped) → dəyişənin qiymətini yeniləyir.

    if loss < best_loss:
      # Əgər bu iterasiyada loss indiyədək ən kiçik olubsa:

      # Update best loss and best image from total loss.
      # Yuxarıdakı ingilis dilindəki şərh: "ən yaxşı loss və şəkli ümumi loss-a görə yeniləyirik".
      best_loss = loss
      best_img = deprocess_img(init_image.numpy())
      # Ən yaxşı loss və ona uyğun şəkli saxlayırıq.
      #
      # init_image.numpy() → tf.Variable-ı adi NumPy array-ə çevirir.
      # deprocess_img(...) → onu normal RGB (0–255) şəklə qaytarır.

      # Use the .numpy() method to get the concrete numpy array
      # Yuxarıdakı ingilis dilindəki şərh: ".numpy() metodu ilə real NumPy array alırıq".
      plot_img = deprocess_img(init_image.numpy())
      imgs.append(plot_img)
      print('Iteration: {}'.format(i))
      # Eyni şəkli ikinci dəfə də deprocess edib imgs siyahısına əlavə edir.
      # (best_img ilə eynidir, təkrar hesablamadır. imgs siyahısı sonra istifadə olunmur.)
      #
      # Hansı iterasiyada yaxşılaşma olduğunu çap edir.
      # Loss azaldıqca bu mesajlar ekranda görünəcək.


plt.imshow(best_img)
print(best_loss)
# Döngü bitəndən sonra:
#
# plt.imshow(best_img) → ən yaxşı nəticəni ekranda göstərir
# print(best_loss)     → ən kiçik loss qiymətini çap edir
#
# Qeyd: best_loss tf.Tensor olduğu üçün çap zamanı
# "tf.Tensor(12345.0, shape=(), dtype=float32)" şəklində görünə bilər.


image = Image.fromarray(best_img.astype('uint8'), 'RGB')
# NumPy array-ini yenidən PIL şəklinə çevirir.
#
# astype('uint8') → qiymətlər tam ədəd (0–255) olmalıdır
# 'RGB'           → 3 kanallı rəngli şəkil formatı

image.save("result.jpg")
# Şəkli Colab-ın müvəqqəti yaddaşında result.jpg adı ilə saxlayır.

files.download("result.jpg")
# Həmin faylı brauzer vasitəsilə kompüterinə yükləyir.
#
# Colab-ın müvəqqəti yaddaşı sessiya bitəndə silinir,
# ona görə nəticəni yükləmək vacibdir.

## Umumi Iazh

Bu kod iki şəkildən üçüncü, yeni bir şəkil yaradır: **birinci şəklin məzmununu, ikinci şəklin üslubu ilə.**

**Ümumi məntiq:**

1. **Hazır model götürülür.** İmageNet-də öyrədilmiş VGG19 yüklənir və dondurulur. Onu nə öyrədirik, nə dəyişirik. O yalnız şəkilə baxıb "bu şəkildə nə var, necə görünür" ölçən bir alət kimi işləyir.

2. **İki hədəf müəyyən edilir.**
   - Məzmun hədəfi: məzmun şəklinin dərin qatdakı (`block5_conv2`) xüsusiyyətləri. Yəni "şəkildə nə var".
   - Üslub hədəfi: üslub şəklinin 5 qatdakı Gram matrisləri. Yəni "rənglər, teksturalar, fırça vuruşları necə birlikdə görünür".

3. **Başlanğıc şəkil seçilir.** Məzmun şəklinin surəti götürülür və bu surət dəyişdiriləcək şəkil olur.

4. **Döngü 100 dəfə təkrarlanır.** Hər addımda:
   - Dəyişdirilən şəkil VGG-dən keçirilir.
   - İki fərq ölçülür: məzmun hədəfindən nə qədər uzaqdır (məzmun loss) və üslub hədəfindən nə qədər uzaqdır (üslub loss). Cəmi ümumi loss-dur.
   - Qradiyent hesablanır: "loss kiçilsin deyə hər pikseli hansı istiqamətdə dəyişək?"
   - Adam optimizer pikselləri bir az dəyişir.
   - Piksel qiymətləri icazə verilən aralıqda saxlanılır.
   - Loss indiyədək ən kiçikdirsə, bu şəkil "ən yaxşı" kimi yadda saxlanılır.

5. **Nəticə saxlanılır.** Ən aşağı loss-lu şəkil normal RGB formatına qaytarılır, ekranda göstərilir və `result.jpg` kimi yüklənir.

**Qısa desək:** model öyrənmir, şəklin özü "öyrənir". Şəkil tədricən elə dəyişir ki, VGG-nin gözündə həm məzmun şəklinə (nə var), həm də üslub şəklinə (necə görünür) oxşasın.

**Çəkilərin rolu:** `content_weight` və `style_weight` tarazlığı müəyyən edir. Məzmun çəkisi böyükdürsə nəticə orijinala daha yaxın, üslub çəkisi böyükdürsə rəsmə daha yaxın olur.

Bir dəqiqləşdirmə: yeni şəkil "iki şəklin birləşməsi" kimi hazır başlamır, **məzmun şəklinin surətindən başlayır** və tədricən dəyişir.

**Addım 1 (bir dəfə, döngüdən əvvəl):**
- Məzmun şəklindən xüsusiyyətlər alınır → saxlanılır (hədəf 1)
- Üslub şəklindən xüsusiyyətlər alınır → Gram matrisinə çevrilib saxlanılır (hədəf 2)

**Addım 2 (hər iterasiyada):**
- Dəyişən şəkil (`init_image`) VGG-dən keçir, ondan da xüsusiyyətlər alınır.
- Sənin dediyin müqayisə aparılır:
  - Onun **məzmun xüsusiyyətləri** məzmun şəklinin xüsusiyyətlərindən nə qədər fərqlənir? → məzmun loss
  - Onun **üslub xüsusiyyətləri (Gram)** üslub şəklinin üslub xüsusiyyətlərindən nə qədər fərqlənir? → üslub loss
- İkisi toplanır → ümumi loss

**Addım 3:** Optimizer şəkli elə dəyişir ki, bu iki fərq kiçilsin. Sonra yenə Addım 2.

Yəni sənin cümləni dəqiq belə yazmaq olar:

> Əvvəl iki orijinal şəklin xüsusiyyətləri alınır. Sonra yaradılan şəkildən də xüsusiyyətlər alınır və hər biri müvafiq orijinalın xüsusiyyətləri ilə müqayisə olunur: məzmun məzmun şəklinin məzmunu ilə, üslub üslub şəklinin üslubu ilə. Fərq nə qədər kiçikdirsə, nəticə o qədər yaxşıdır.

Burada "hər birinin öz xüsusiyyəti" dedikdə: məzmun yalnız məzmun şəklindən, üslub yalnız üslub şəklindən götürülür. Yaradılan şəkil məzmunu üslub şəklindən, üslubu məzmun şəklindən almır.